# Data on the Line · Setup test (5 minutes)

This notebook checks that the environment you'll use in the session is ready. You'll follow the same steps in the case.

1. Open the notebook in Colab with the link or QR code you were given. Colab downloads the data files from GitHub by itself.
2. Run all the cells in order with *Runtime → Run all*. At the end you should see **"All set."**

If you'd rather use Jupyter on your own computer, download the `Setup_Test` folder and open the notebook there.

If anything fails, send the error message to your instructor before the session.

In [ ]:
# on Colab the data files come from GitHub automatically; on your own computer this is skipped
import importlib
import os
import shutil
import subprocess
import sys
import zipfile
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from pathlib import Path

REPO_URL = "https://github.com/burcu-koca-kd/data-on-the-line.git"
REPO = Path("/content/data-on-the-line")
TEST_DIR = REPO / "en/1_Preparation/Setup_Test"


def _clone():
    try:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(REPO)],
                       check=True, capture_output=True)
    except subprocess.CalledProcessError:
        print("✗ The files couldn't be downloaded from GitHub. Check your internet "
              "connection, then open the notebook link again or rerun this cell.")
        raise RuntimeError("Files not downloaded (see the message above).") from None


def _update():
    """Brings the downloaded copy up to date with GitHub; leaves your own files alone."""
    git = ["git", "-C", str(REPO)]
    fetched = subprocess.run(git + ["fetch", "-q", "--depth", "1"], capture_output=True)
    if fetched.returncode == 0:
        subprocess.run(git + ["reset", "-q", "--hard", "origin/main"], capture_output=True)


if "google.colab" in sys.modules:
    # an earlier copy is brought up to date; if it is incomplete, it is downloaded again
    if REPO.exists():
        _update()
    if not TEST_DIR.exists():
        shutil.rmtree(REPO, ignore_errors=True)
        _clone()
    os.chdir(TEST_DIR)
print("Working folder:", Path.cwd())
if Path("data").exists():
    print("data folder found ✓")
else:
    print("data folder NOT found: check the folder path.")

In [ ]:
def _install(*packages):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f"pip install {' '.join(packages)} failed. End of the error message:")
        print(result.stderr[-1500:])


def _pulp_major():
    try:
        return int(version("pulp").split(".")[0])
    except PackageNotFoundError:
        return 0


def _forget_pulp():
    """Drops PuLP from memory so that the next import sees newly installed packages."""
    for name in [m for m in sys.modules if m == "pulp" or m.startswith("pulp.")]:
        del sys.modules[name]
    importlib.invalidate_caches()


# Colab has either no PuLP or version 4; the case is written for version 3
if not 2 <= _pulp_major() < 4:
    _install("pulp>=2.7,<4")
    _forget_pulp()
for module, package in (("statsmodels", "statsmodels"), ("sklearn", "scikit-learn")):
    if find_spec(module) is None:
        _install(package)

import numpy as np
import pandas as pd
import pulp
import sklearn
import statsmodels

PULP_OK = 2 <= _pulp_major() < 4
if not PULP_OK:
    print("⚠ PuLP 3 could not be installed and version 4 is still loaded. Choose "
          "Runtime → Restart session and run this notebook again from the top.")
print("pulp", pulp.__version__, "✓" if PULP_OK else "✗", "| pandas", pd.__version__,
      "✓ | statsmodels", statsmodels.__version__, "✓ | scikit-learn", sklearn.__version__, "✓")

In [ ]:
result = {"PuLP version is suitable (version 3 or earlier)": PULP_OK}
result["Reading a data file"] = len(pd.read_csv("data/sample.csv")) == 3

# you will do exactly this at minutes 45 and 95
with zipfile.ZipFile("release_test.zip") as z:
    z.extractall(".", pwd=b"test")
result["Unlocking a password-protected file"] = Path("release/test.csv").exists()


def solves(solver):
    """Checks whether the given solver can solve a tiny model; returns True if it can."""
    try:
        m = pulp.LpProblem("test", pulp.LpMinimize)
        x = pulp.LpVariable.dicts("x", range(3), cat="Binary")
        m += pulp.lpSum(x.values())
        m += x[0] + x[1] >= 1
        m += x[1] + x[2] >= 1
        m.solve(solver)
        return pulp.LpStatus[m.status] == "Optimal" and pulp.value(m.objective) == 1
    except Exception:
        return False


solver = "CBC" if solves(pulp.PULP_CBC_CMD(msg=False)) else None
# backup solver: installed only if CBC won't start (on some Macs, for example)
if solver is None and find_spec("highspy") is None:
    _install("highspy")
    _forget_pulp()
    import pulp
if solver is None and hasattr(pulp, "HiGHS") and solves(pulp.HiGHS(msg=False)):
    solver = "HiGHS"
label = "Solving a small integer model" + (f" ({solver})" if solver else "")
result[label] = solver is not None

for name, ok in result.items():
    print(("✓ " if ok else "✗ ") + name)
if not solver:
    print()
    print("The solver would not start. The easy fix is to use Colab. If you want to"
          " work on your own Mac, run 'pip install highspy' in Terminal, restart "
          "the kernel and run this notebook again from the top.")
print()
if all(result.values()):
    print("All set.")
else:
    print("A step failed: send this message to your instructor.")